In [1]:
import sys
sys.path.append(r'C:\Users\admin\Fusion\Fusion-0.7.0')

import numpy as np
import pandas as pd

from sklearn.metrics import make_scorer
from sklearn.metrics import roc_auc_score

from fusion.selection import CountChunker, TimeSeriesChunker
from fusion.selection import VInformationSelector
from fusion.selection import IVInformationSelector
from fusion.selection import CNSTStabilitySelector
from fusion.selection import CStabilitySelector
from fusion.selection import PSIStabilitySelector
from fusion.selection import KSStabilitySelector
from fusion.selection import GCorrelationSelector
from fusion.selection import IWrappSelector

In [2]:
df = pd.read_csv('df_for_selecting_feats.csv')

In [3]:
THRESHOLDS = {
    'VAR': 0.9,
    'IV': 0.1,
    'CONST': 0.05,
    'PSI': 0.25,
    'AVG': 0.25,
    'KS': 0.15,
    'CORR': 0.85}

In [4]:
TARGET = 'target'
KEY = 'demand'
INFORMATIVE = 'oot'
COUNTS = len([KEY, TARGET, INFORMATIVE])

### Отбор признаков

Отбор проводится по факторам информационной ценности, стабильности и корреляции.

In [5]:
chunker = CountChunker(
    count=5,
    scheme='all-next')

In [6]:
def rule(results: list, threshold: float):
    values = []
    for value in results:
        values += [value if isinstance(value, (float, np.float32)) else getattr(value, 'value')]

    return np.mean(values) < threshold

In [7]:
def types(frame: pd.DataFrame, thresholds: int):
    categorical, numerical = [], []
    for name in frame.columns:
        if frame[name].unique().size <= thresholds:
            categorical += [name]
        else:
            numerical += [name]

    return numerical, categorical

In [8]:
def typing(frame: pd.DataFrame):
    for name in frame.select_dtypes(include=['int64', 'float64']).columns:
        frame[name] = pd.to_numeric(frame[name], downcast='float')
        frame[name] = pd.to_numeric(frame[name], downcast='integer')

    return frame

In [9]:
def gini_score(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

#### Селекторы

In [10]:
numerical, categorical = types(df.iloc[:, COUNTS:], 30)
len(numerical), len(categorical)

(42, 13)

In [12]:
selector = VInformationSelector(
    threshold=THRESHOLDS['VAR'],
    nan='ignore')

In [13]:
result = selector.select(df.iloc[:, COUNTS:])

In [14]:
result.report.sort_values(by='UNIQUE', ascending=False)

,UNIQUE,PASS
FEATURES,,
appeal_last_lk,0.996217,False
appeal_last_em,0.995782,False
all_appeal_lk,0.994245,False
avg_appeal_lk,0.994245,False
all_appeal_em,0.992145,False
avg_appeal_em,0.992145,False
call_cnt_last,0.971447,False
dop_ratio,0.947661,False
dop_first_service_ratio,0.921158,False


In [15]:
len(selector.excluded)

9

In [16]:
selector = IVInformationSelector(
    threshold=THRESHOLDS['IV'],
    binning='quantiles',
    nan='drop',
    jobs=1)

In [17]:
%%time
result = selector.select(df.iloc[:, COUNTS:], y=df[TARGET], categories=categorical)

CPU times: total: 11.7 s
Wall time: 11.8 s


In [18]:
result.report.sort_values(by='IV', ascending=False)

,BIN 1,BIN 2,BIN 3,BIN 4,BIN 5,BIN 6,BIN 7,BIN 8,BIN 9,BIN 10,...,BIN 15,BIN 16,BIN 17,BIN 18,BIN 19,BIN 20,BIN 21,BIN 22,IV,PASS
FEATURES,,,,,,,,,,,,,,,,,,,,,
ltv,1.528553,0.684455,0.330553,-0.025995,-0.177766,-0.358841,-0.533134,-0.637418,-0.729116,-0.962229,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.529444,True
days_after_order_time,0.983467,0.850463,0.330877,0.117524,-0.087273,-0.231481,-0.290768,-0.545927,-0.921143,-1.286774,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.444238,True
period30_ratio,-1.216318,-0.821079,-0.654077,0.434463,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.373751,True
comis_to_paym_ratio,1.343414,0.325945,-0.478315,-0.469995,-0.486977,-0.444640,-0.315963,-0.224037,0.136408,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.359459,True
comis_to_paym_ratio_max,1.343403,0.325997,-0.478264,-0.469944,-0.486926,-0.444588,-0.315911,-0.223985,0.136459,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.359394,True
comis_to_paym_ratio_last,1.200440,0.629311,-0.461835,-0.444443,-0.426525,-0.362198,-0.320975,-0.290570,-0.386232,-0.191598,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.355396,True
order_number,0.347921,-0.688947,-0.901985,-1.060825,-1.423726,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.348413,True
comis_to_debt_ratio_max,1.308920,0.349515,-0.152447,-0.170443,-0.331515,-0.461256,-0.433447,-0.373034,-0.237404,-0.096843,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.302845,True
comis_to_debt_ratio,1.308920,0.349515,-0.152447,-0.170443,-0.331515,-0.461256,-0.433447,-0.373034,-0.237404,-0.096843,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.302845,True


In [19]:
selector = CNSTStabilitySelector(
    threshold=THRESHOLDS['CONST'],
    rule=rule,
    chunker=chunker,
    jobs=5)

In [20]:
%%time
result = selector.select(df.iloc[:, COUNTS:])

CPU times: total: 188 ms
Wall time: 2.15 s


In [21]:
result.report.sort_values(by='MAX', ascending=False)

TESTS,TEST 1,TEST 2,TEST 3,TEST 4,TEST 5,TEST 6,TEST 7,TEST 8,TEST 9,TEST 10,MIN,MEAN,MAX,PASS
FEATURES,,,,,,,,,,,,,,
avg_time_btw_order,0.005693,8.750341e-02,0.012295,0.026593,9.319645e-02,0.017988,0.020899,0.075208,0.114096,0.038888,0.005693,0.049236,0.114096,True
time_btw_last_order,0.005693,8.748785e-02,0.012295,0.026593,9.318090e-02,0.017988,0.020899,0.075193,0.114080,0.038888,0.005693,0.049230,0.114080,True
comis_to_paym_ratio_last,0.000202,3.421994e-04,0.000684,0.000684,5.444113e-04,0.000887,0.000887,0.000342,0.000342,0.000000,0.000000,0.000492,0.000887,True
amount_inc_ratio_last,0.000202,2.644309e-04,0.000264,0.000264,6.221904e-05,0.000062,0.000062,0.000000,0.000000,0.000000,0.000000,0.000118,0.000264,True
income,0.000202,2.644309e-04,0.000264,0.000264,6.221904e-05,0.000062,0.000062,0.000000,0.000000,0.000000,0.000000,0.000118,0.000264,True
dops_inc_ratio_last,0.000202,2.644309e-04,0.000264,0.000264,6.221904e-05,0.000062,0.000062,0.000000,0.000000,0.000000,0.000000,0.000118,0.000264,True
start_amount_inc_ratio_last,0.000202,2.644309e-04,0.000264,0.000264,6.221904e-05,0.000062,0.000062,0.000000,0.000000,0.000000,0.000000,0.000118,0.000264,True
dops_inc_ratio,0.000109,1.555476e-04,0.000156,0.000156,4.666428e-05,0.000047,0.000047,0.000000,0.000000,0.000000,0.000000,0.000072,0.000156,True
avg_start_amount_inc_ratio,0.000109,1.555476e-04,0.000156,0.000156,4.666428e-05,0.000047,0.000047,0.000000,0.000000,0.000000,0.000000,0.000072,0.000156,True


In [22]:
len(selector.excluded)

0

In [23]:
selector.exclude(df, inplace=True)

In [24]:
selector = CStabilitySelector(
    function=lambda base, test: np.abs(np.mean(base) - np.mean(test)) / np.std(base),
    threshold=THRESHOLDS['AVG'],
    rule=rule,
    chunker=chunker,
    nan='drop',
    jobs=5)

In [25]:
%%time
result = selector.select(
    x=df[[feature for feature in df.iloc[:, COUNTS:].columns if feature in numerical]])

CPU times: total: 141 ms
Wall time: 796 ms


In [26]:
result.report.sort_values(by='MAX', ascending=False)

TESTS,TEST 1,TEST 2,TEST 3,TEST 4,TEST 5,TEST 6,TEST 7,TEST 8,TEST 9,TEST 10,MIN,MEAN,MAX,PASS
FEATURES,,,,,,,,,,,,,,
dops_inc_ratio,0.004238,0.193337,0.927936,0.914885,10.198994,48.119635,47.445967,0.055815,0.054823,0.000447,0.000447,10.791608,48.119635,False
avg_start_amount_inc_ratio,0.055240,0.121903,0.541928,0.733099,0.010887,0.079484,0.110705,0.055530,0.080804,0.012135,0.010887,0.180172,0.733099,True
income,0.067652,0.211744,0.679331,0.690688,0.135351,0.574569,0.585238,0.418271,0.428430,0.010627,0.010627,0.380190,0.690688,False
start_amount_inc_ratio_last,0.032804,0.115699,0.515661,0.456411,0.022596,0.131623,0.115472,0.071730,0.061104,0.004815,0.004815,0.152791,0.515661,True
dops_inc_ratio_last,0.021047,0.176146,0.497920,0.422938,0.056303,0.173111,0.145892,0.043910,0.033678,0.006109,0.006109,0.157705,0.497920,True
comis_to_paym_ratio_max,0.351609,0.396671,0.457695,0.417926,0.046443,0.109335,0.068349,0.050389,0.017551,0.036178,0.017551,0.195214,0.457695,True
comis_to_paym_ratio,0.351546,0.396600,0.457512,0.417749,0.046436,0.109216,0.068234,0.050301,0.017465,0.036178,0.017465,0.195124,0.457512,True
avg_pl,0.162988,0.288538,0.377502,0.448312,0.160544,0.274303,0.364849,0.145296,0.260943,0.141281,0.141281,0.262456,0.448312,False
avg_amount_inc_ratio,0.010859,0.103076,0.357255,0.434202,0.056621,0.212686,0.259931,0.039324,0.051229,0.007236,0.007236,0.153242,0.434202,True


In [27]:
len(selector.excluded)

3

In [28]:
selector.exclude(df, inplace=True)

In [29]:
selector = KSStabilitySelector(
    threshold=THRESHOLDS['KS'],
    rule=rule,
    chunker=chunker,
    nan='drop',
    jobs=5)

In [30]:
%%time
result = selector.select(
    x=df[[feature for feature in df.iloc[:, COUNTS:].columns if feature in numerical]])

CPU times: total: 156 ms
Wall time: 1.71 s


In [31]:
result.report.sort_values(by='MAX', ascending=False)

TESTS,TEST 1,TEST 2,TEST 3,TEST 4,TEST 5,TEST 6,TEST 7,TEST 8,TEST 9,TEST 10,MIN,MEAN,MAX,PASS
FEATURES,,,,,,,,,,,,,,
avg_amount_inc_ratio,0.016518,0.052946,0.236716,0.267391,0.042478,0.226062,0.256737,0.185882,0.209059,0.030674,0.016518,0.152446,0.267391,False
comis_to_paym_ratio_max,0.189388,0.212743,0.253463,0.215773,0.092185,0.093337,0.045152,0.040720,0.047625,0.047101,0.040720,0.123749,0.253463,True
comis_to_paym_ratio,0.189379,0.212733,0.253425,0.215735,0.092181,0.093303,0.045111,0.040692,0.047614,0.047101,0.040692,0.123727,0.253425,True
amount_inc_ratio_last,0.014789,0.058102,0.225816,0.249988,0.047656,0.215370,0.239542,0.167714,0.191887,0.024172,0.014789,0.143504,0.249988,True
avg_start_amount_inc_ratio,0.003567,0.039709,0.216180,0.208869,0.039414,0.218822,0.205494,0.184047,0.180127,0.008089,0.003567,0.130432,0.218822,True
dops_inc_ratio_last,0.039448,0.050542,0.195965,0.212049,0.029061,0.180279,0.190569,0.163312,0.164416,0.020439,0.020439,0.124608,0.212049,True
avg_cnt_session,0.040333,0.122017,0.170953,0.198205,0.084188,0.136452,0.157872,0.048936,0.076188,0.044456,0.040333,0.107960,0.198205,True
start_amount_inc_ratio_last,0.007418,0.042205,0.196708,0.189163,0.038504,0.195976,0.188432,0.171074,0.163545,0.012615,0.007418,0.120564,0.196708,True
comis_to_debt_ratio_max,0.126740,0.167000,0.193772,0.176459,0.093006,0.090627,0.062395,0.037721,0.060120,0.032977,0.032977,0.104082,0.193772,True


In [32]:
len(selector.excluded)

1

In [33]:
selector.exclude(df, inplace=True)

In [34]:
selector = PSIStabilitySelector(
    threshold=THRESHOLDS['PSI'],
    rule=rule,
    chunker=chunker,
    binning='quantiles',
    nan='drop',
    jobs=5)

In [35]:
%%time
result = selector.select(df.iloc[:, COUNTS:], categories=categorical)

CPU times: total: 219 ms
Wall time: 12.9 s


In [36]:
result.report.sort_values(by='MAX', ascending=False)

TESTS,TEST 1,TEST 2,TEST 3,TEST 4,TEST 5,TEST 6,TEST 7,TEST 8,TEST 9,TEST 10,MIN,MEAN,MAX,PASS
FEATURES,,,,,,,,,,,,,,
comis_to_paym_ratio_max,0.242388,0.281959,0.419835,0.340769,0.088929,7.934452e-02,0.021864,0.034294,0.049219,2.392521e-02,2.186375e-02,0.158253,0.419835,True
comis_to_paym_ratio,0.242372,0.281907,0.419733,0.340696,0.088909,7.930206e-02,0.021836,0.034311,0.049250,2.392521e-02,2.183556e-02,0.158224,0.419733,True
dops_inc_ratio_last,0.002122,0.016721,0.402844,0.332165,0.009403,3.704136e-01,0.299605,0.299356,0.239599,3.497029e-03,2.122074e-03,0.197572,0.402844,True
amount_inc_ratio_last,0.002012,0.020076,0.355403,0.398611,0.011783,3.307891e-01,0.366304,0.232400,0.248055,5.892686e-03,2.012374e-03,0.197133,0.398611,True
avg_start_amount_inc_ratio,0.000615,0.010170,0.382097,0.328012,0.009964,3.896933e-01,0.334466,0.291210,0.246880,2.263395e-03,6.146853e-04,0.199537,0.389693,True
start_amount_inc_ratio_last,0.000890,0.012493,0.309985,0.266817,0.008787,3.085201e-01,0.265182,0.242187,0.193660,2.808245e-03,8.898367e-04,0.161133,0.309985,True
pl_last,0.027119,0.110714,0.199384,0.299057,0.029789,8.302833e-02,0.156683,0.014351,0.053449,1.369645e-02,1.369645e-02,0.098727,0.299057,True
days_after_order_time,0.014585,0.059574,0.139408,0.253055,0.043436,1.350990e-01,0.285589,0.029661,0.106866,3.057391e-02,1.458537e-02,0.109785,0.285589,True
comis_to_debt_ratio_max,0.097156,0.200487,0.261579,0.225726,0.094747,6.780814e-02,0.027229,0.023654,0.061629,1.901237e-02,1.901237e-02,0.107903,0.261579,True


In [37]:
len(selector.excluded)

0

In [38]:
selector.exclude(df, inplace=True)

In [39]:
selector = GCorrelationSelector(
    threshold=THRESHOLDS['CORR'],
    method='pearson',
    nan='drop')

In [40]:
%%time
result = selector.select(df.iloc[:, COUNTS:])

CPU times: total: 1.06 s
Wall time: 1.06 s


In [41]:
result.report

FEATURES,ltv_last,ltv_avg,comis_to_debt_ratio_max,comis_to_debt_ratio,comis_to_debt_ratio_last,comis_to_paym_ratio_max,comis_to_paym_ratio,comis_to_paym_ratio_last,dop_second_service_ratio,dop_first_service_ratio,...,dop_sum_ratio,ltv,days_after_order_time,order_number,delay_ratio,avg_fact_order_sum,time_btw_last_order,lifetime_start_last,lifetime_last,order_time_last_ratio
FEATURES,,,,,,,,,,,,,,,,,,,,,
ltv_last,1.000000,0.852684,0.285689,0.285689,0.319078,0.128389,0.128525,0.072961,0.089202,0.094559,...,-0.214602,0.517245,0.277139,-0.013999,-0.326343,0.388250,-0.038930,0.213228,0.686901,0.629185
ltv_avg,0.852684,1.000000,0.353162,0.353162,0.318803,0.157801,0.157960,0.114183,0.091695,0.118479,...,-0.232106,0.589716,0.338641,-0.014264,-0.439727,0.423146,-0.019919,0.212139,0.606251,0.566463
comis_to_debt_ratio_max,0.285689,0.353162,1.000000,1.000000,0.876436,0.830413,0.830424,0.659258,0.173391,0.232945,...,0.502675,0.170574,0.252235,-0.003428,-0.383570,-0.378694,0.016886,0.107987,0.432427,0.421682
comis_to_debt_ratio,0.285689,0.353162,1.000000,1.000000,0.876436,0.830413,0.830424,0.659258,0.173391,0.232945,...,0.502675,0.170574,0.252235,-0.003428,-0.383570,-0.378694,0.016886,0.107987,0.432427,0.421682
comis_to_debt_ratio_last,0.319078,0.318803,0.876436,0.876436,1.000000,0.737979,0.737996,0.744527,0.145128,0.201929,...,0.442405,0.190125,0.268194,0.040869,-0.334121,-0.339283,0.017506,0.109705,0.476556,0.457867
comis_to_paym_ratio_max,0.128389,0.157801,0.830413,0.830413,0.737979,1.000000,1.000000,0.821987,0.096188,0.314554,...,0.403261,0.146294,0.203277,0.130792,-0.104904,-0.333927,-0.005513,0.091276,0.175689,0.164659
comis_to_paym_ratio,0.128525,0.157960,0.830424,0.830424,0.737996,1.000000,1.000000,0.821987,0.096157,0.314535,...,0.403247,0.146294,0.203105,0.130820,-0.104743,-0.333930,-0.005513,0.091275,0.174908,0.163751
comis_to_paym_ratio_last,0.072961,0.114183,0.659258,0.659258,0.744527,0.821987,0.821987,1.000000,0.031242,0.259649,...,0.321262,0.159408,0.215602,0.181149,-0.063059,-0.278601,0.001250,0.079792,0.123051,0.111979
dop_second_service_ratio,0.089202,0.091695,0.173391,0.173391,0.145128,0.096188,0.096157,0.031242,1.000000,0.295173,...,0.432664,-0.293897,-0.279385,-0.464146,-0.110231,-0.087244,0.124364,-0.069204,0.043490,0.056338


In [42]:
len(selector.excluded)

11

In [43]:
selector.exclude(df, inplace=True)

In [44]:
df

,demand,target,oot,ltv_avg,comis_to_debt_ratio_last,comis_to_paym_ratio,comis_to_paym_ratio_last,dop_second_service_ratio,dop_first_service_ratio,dop_ratio,...,avg_approve,dop_sum_ratio,ltv,days_after_order_time,order_number,delay_ratio,avg_fact_order_sum,time_btw_last_order,lifetime_start_last,order_time_last_ratio
0,23809874,1,0,8144.91,0.29,0.09,0.09,1.0,1.0,1.0,...,1.0,0.600,8144.914,69,1,0.0,4000.000,NaN,22,3.181818
1,23810121,1,0,4923.76,0.16,0.10,0.10,1.0,1.0,1.0,...,1.0,0.300,4923.763,16,1,1.0,8000.000,NaN,16,1.062500
2,23845577,0,0,3257.54,0.29,0.14,0.14,1.0,1.0,1.0,...,1.0,0.800,3257.543,5,1,1.0,3000.000,NaN,16,0.375000
3,24846191,0,0,4387.87,0.16,0.11,0.09,1.0,1.0,1.0,...,1.0,0.436,8775.745,53,2,1.0,5500.000,25.0,26,0.923077
4,30554758,1,0,4193.63,0.20,0.11,0.11,1.0,1.0,1.0,...,1.0,0.450,12580.888,211,3,1.0,5333.333,150.0,16,0.562500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321437,46765176,0,0,3193.60,0.08,0.06,0.06,1.0,1.0,1.0,...,1.0,0.240,3193.600,7,1,1.0,10000.000,NaN,30,0.233333
321438,48222616,0,0,3653.12,0.13,0.08,0.09,1.0,1.0,1.0,...,1.0,0.240,7306.243,38,2,1.0,10000.000,26.0,21,0.238095
321439,49464215,0,0,3693.07,0.13,0.08,0.09,1.0,1.0,1.0,...,1.0,0.240,11079.205,62,3,1.0,10000.000,24.0,19,0.000000
321440,49512219,0,0,3734.27,0.13,0.09,0.09,1.0,1.0,1.0,...,1.0,0.240,14937.088,65,4,1.0,10000.000,1.0,18,0.111111


In [45]:
df.to_csv('df_for_analyzing_feats.csv', index=False)